### Bem-vindo ao Dia 3 da Semana 6!

Vamos experimentar muitos outros servidores MCP


In [13]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)

True

### O primeiro tipo de servidor MCP: roda localmente, tudo local

Aqui esta um bem interessante: uma memoria baseada em grafo de conhecimento.

E uma memoria persistente de entidades, observacoes sobre elas, e relacionamentos entre elas.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory


In [14]:
params = {"command": "npx","args": ["-y", "mcp-memory-libsql"],"env": {"LIBSQL_URL": "file:./memory/ed.db"}}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='create_entities', description='Create new entities with observations and optional embeddings', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'entityType': {'type': 'string'}, 'observations': {'type': 'array', 'items': {'type': 'string'}}, 'embedding': {'type': 'array', 'items': {'type': 'number'}}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities']}, annotations=None, title='Create new entities with observations and optional embeddings'),
 Tool(name='search_nodes', description='Search for entities and their relations using text or vector similarity', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'query': {'anyOf': [{'type': 'string'}, {'type': 'array', 'items': {'type': 'number'}}]}}, 'required': ['query']}, annotations=None, title='Search

In [15]:
instructions = "Voce usa suas ferramentas de entidade como uma memoria persistente para armazenar e lembrar informacoes sobre suas conversas."
request = "Meu nome e Ed. Sou um engenheiro de LLM. Estou ministrando um curso sobre agentes de IA, incluindo o incrivel protocolo MCP. \
MCP e um protocolo para conectar agentes com ferramentas, recursos e modelos de prompt, e facilita integrar agentes de IA com capacidades."
model = "gpt-5-mini"


In [16]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Perfeito — obrigado, Ed. Salvei estas informações:

- Nome: Ed  
- Profissão: Engenheiro de LLM  
- Está ministrando um curso sobre agentes de IA  
- Enfoca o protocolo MCP (protocolo para conectar agentes com ferramentas, recursos e modelos de prompt)

Quer que eu salve algo mais sobre o curso (tópicos, público-alvo, datas) ou como você quer que eu use essa memória nas próximas interações?

In [17]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "Meu nome e Ed. O que voce sabe sobre mim?")
    display(Markdown(result.final_output))

Atualmente eu sei o seguinte sobre você (segundo minha memória):

- Nome: Ed.  
- Profissão: Engenheiro de LLM.  
- Está ministrando um curso sobre agentes de IA.  
- Está ensinando sobre o protocolo MCP (protocolo para conectar agentes com ferramentas, recursos e modelos de prompt).

Quer que eu acrescente ou corrija alguma coisa (por exemplo pronomes, cidade, preferências, nível do curso, tópicos que quer enfatizar)? Posso também apagar qualquer item se preferir.

### Verifique o trace:

https://platform.openai.com/traces


### O segundo tipo de servidor MCP - roda localmente, chama um servico web

### Brave Search - desculpe - isso vai precisar de outra chave de API! Mas e gratuita de novo.

https://brave.com/search/api/

Configure sua conta e coloque sua chave no .env sob `BRAVE_API_KEY`


In [18]:
env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}
params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": env}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='brave_web_search', description='Performs a web search using the Brave Search API, ideal for general queries, news, articles, and online content. Use this for broad information gathering, recent events, or when you need diverse web sources. Supports pagination, content filtering, and freshness controls. Maximum 20 results per request, with offset for pagination. ', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query (max 400 chars, 50 words)'}, 'count': {'type': 'number', 'description': 'Number of results (1-20, default 10)', 'default': 10}, 'offset': {'type': 'number', 'description': 'Pagination offset (max 9, default 0)', 'default': 0}}, 'required': ['query']}, annotations=None),
 Tool(name='brave_local_search', description="Searches for local businesses and places using Brave's Local Search API. Best for queries related to physical locations, businesses, restaurants, services, etc. Returns detailed information including:\

In [19]:
instructions = "Voce consegue pesquisar na web por informacoes e resumir rapidamente os principais pontos."
request = f"Por favor, pesquise as ultimas noticias sobre o preco das acoes da Amazon e resuma rapidamente as perspectivas. \
Para contexto, a data atual e {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-5-mini"


In [20]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Aqui vai um resumo rápido (com referências) sobre o preço das ações da Amazon (AMZN) e as perspectivas atuais — data: 2025-10-03.

Preço e movimento recente
- Cotação aproximada em 03/10/2025: US$ 219–220 por ação (Investing.com mostrou US$ 219,65; fechamento anterior ~US$ 222,41). Fonte: Investing.com (03/10/2025).

Principais fatores que estão afetando o preço
- Grandes investimentos em IA e capex: a Amazon vem anunciando gastos muito altos para infraestrutura de IA (relatos de ordem de US$100 bi em capex para 2025) e investimentos em parceiros/tecnologia de IA. Isso é visto como potencial catalisador de longo prazo, mas também como pressão de custo no curto prazo. (CNBC / Fortune / comunicados da empresa, 2025).
- AWS e competição na nuvem: preocupações sobre desaceleração ou crescimento menos robusto do AWS e competição crescente (Microsoft, Google, Oracle) reduzem o apelo do papel para alguns investidores. (Bloomberg 02/10/2025; Reuters).
- Resultados e reação do mercado: resultados trimestre a trimestre e orientação continuam a gerar picos de volatilidade — gastos com IA/infraestrutura podem fazer com que lucro e margem fiquem sob escrutínio mesmo quando receita cresce. (várias coberturas de resultados 2025).
- Notícias operacionais: saída recente de executivo sênior na área de dispositivos após lançamentos (Reuters, 03/10/2025) — sinal a acompanhar para unidade de hardware/dispositivos.

Perspectivas (curto, médio e longo prazo)
- Curto prazo (semanas a trimestres): expectativa de volatilidade em torno de resultados e comunicados sobre capex/growth do AWS; investidores avaliarão se o ritmo de investimento em IA está comprimindo margens ou pavimenta recuperação futura. Risco de revisões de lucro/guia e reações negativas do mercado.
- Médio prazo (1–2 anos): se a Amazon conseguir traduzir investimentos em IA em produtos/serviços monetizáveis (novos serviços AWS, publicidade mais eficiente, eficiência logística), há potencial de re-rating positivo. Porém, se a concorrência na nuvem corroer preços/participação ou os custos permanecerem altos, a pressão sobre a ação pode continuar.
- Longo prazo (3+ anos): a aposta em IA é vista por muitos analistas como uma opção estratégica significativa — possíveis ganhos substanciais se a execução for forte; mas isso depende de margens, escala e vantagem competitiva sustentável.

Catalisadores a monitorar
- Próximos resultados trimestrais e orientação (particularmente métricas do AWS).
- Atualizações sobre despesas de capital / cronograma de investimentos em IA.
- Lançamentos de produtos AI/serviços AWS (Bedrock/agent offerings, parcerias com startups de IA).
- Indicadores macro (juros, demanda do consumidor) e notícias regulatórias.

Riscos principais
- Competição mais forte no cloud/IA reduzindo receita/margens.
- Capex elevado por longos períodos sem retorno rápido.
- Problemas operacionais em unidades como dispositivos ou publicidade.
- Volatilidade por expectativas de lucro x gasto com tecnologia.

Conclusão curta
- Sentimento atual: misto — otimismo estrutural sobre IA e AWS de longo prazo versus preocupação de curto prazo com custos elevados, competição e resultados trimestrais. A ação tende a permanecer sensível a notícias sobre AWS, gastos com IA e orientações trimestrais.

Se quiser, posso:
- Trazer os links das matérias citadas (Reuters, Bloomberg, Investing, CNBC).
- Monitorar menções a AMZN nas próximas 24–48h e enviar um resumo.
- Fazer uma análise mais aprofundada (valoração, múltiplos, cenário bull/bear).

Obs.: isto é resumo informativo, não é conselho financeiro.

### Como de costume, confira o trace:

https://platform.openai.com/traces


## E agora o terceiro tipo: executando remotamente

Na verdade e bem dificil encontrar um "remote MCP server" aka "hosted MCP server" aka "managed MCP server".

Nao e um modelo comum para usar ou compartilhar servidores MCP, e nao existe uma forma padrao de descobrir servidores MCP remotos.

A Anthropic lista alguns servidores MCP remotos, mas eles sao para aplicacoes pagas com usuarios corporativos:

https://docs.anthropic.com/en/docs/agents-and-tools/remote-mcp-servers

A CloudFlare tem ferramentas para voce criar e implantar seus proprios servidores MCP remotos, mas isso nao parece ser uma pratica comum:

https://developers.cloudflare.com/agents/guides/remote-mcp-server/


# E de volta ao segundo tipo: o servidor MCP da Polygon.io


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">POR FAVOR LEIA!!-</h2>
            <span style="color:#ff7800;">Este servico para dados do mercado financeiro tem um plano GRATUITO e um plano PAGO, e podemos usar qualquer um deles dependendo do seu interesse.
            </span>
        </td>
    </tr>
</table>


## NOVA SECAO: Apresentando a polygon.io

Polygon.io e um provedor de dados financeiros muito popular. Ele tem um plano gratuito e um plano pago. E tambem tem um servidor MCP!

Primeiro, leia sobre polygon.io no excelente site deles, incluindo a pagina de precos:

https://polygon.io

### Polygon.io Parte 1: servico gratuito da Polygon.io (o plano pago sera totalmente opcional, claro!)

1. Por favor, crie uma conta em polygon.io (canto superior direito)
2. Depois de entrar, selecione "Keys" na navegacao da esquerda
3. Pressione o botao azul "New Key"
4. Copie o nome da chave
5. Edite o seu arquivo .env e adicione a linha:

`POLYGON_API_KEY=xxxx`


In [21]:
load_dotenv(override=True)
polygon_api_key = os.getenv("POLYGON_API_KEY")
if not polygon_api_key:
    print("POLYGON_API_KEY nao foi definido")


In [22]:
from polygon import RESTClient
client = RESTClient(polygon_api_key)
client.get_previous_close_agg("AAPL")[0]

PreviousCloseAgg(ticker='AAPL', close=258.02, high=259.24, low=253.95, open=254.665, timestamp=1759521600000, volume=49149679.0, vwap=257.8596)

### Empacotado em um modulo python que faz cache de precos de fim de dia

Eu criei um modulo python `market.py` que usa essa API para buscar precos de acoes.

Mas a API gratuita e bastante limitada em taxa - entao fui um pouco esperto; quando voce pede o preco de uma acao, essa funcao recupera todo o mercado de acoes do fim do dia e armazena em cache no nosso banco de dados.


In [23]:
from market import get_share_price
get_share_price("AAPL")

Was not able to use the polygon API due to {"status":"NOT_AUTHORIZED","request_id":"8bdff9a2dfbd830138b1d866cecd11ac","message":"Attempted to request today's data before end of day. Please upgrade your plan at https://polygon.io/pricing"}; using a random number


63.0

In [24]:
# sem preocupacoes com limite de taxa!

for i in range(1000):
    get_share_price("AAPL")
get_share_price("AAPL")

Was not able to use the polygon API due to {"status":"NOT_AUTHORIZED","request_id":"321644c64054716efa53c590acd603ad","message":"Attempted to request today's data before end of day. Please upgrade your plan at https://polygon.io/pricing"}; using a random number
Was not able to use the polygon API due to HTTPSConnectionPool(host='api.polygon.io', port=443): Max retries exceeded with url: /v2/aggs/ticker/SPY/prev (Caused by ResponseError('too many 429 error responses')); using a random number
Was not able to use the polygon API due to HTTPSConnectionPool(host='api.polygon.io', port=443): Max retries exceeded with url: /v2/aggs/ticker/SPY/prev (Caused by ResponseError('too many 429 error responses')); using a random number
Was not able to use the polygon API due to HTTPSConnectionPool(host='api.polygon.io', port=443): Max retries exceeded with url: /v2/aggs/ticker/SPY/prev (Caused by ResponseError('too many 429 error responses')); using a random number
Was not able to use the polygon API 

KeyboardInterrupt: 

### E eu transformei isso em um servidor MCP

Assim como fizemos com accounts.py; veja `market_server.py`


In [25]:
params = {"command": "uv", "args": ["run", "market_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools

[Tool(name='lookup_share_price', description='This tool provides the current price of the given stock symbol.\n\n    Args:\n        symbol: the symbol of the stock\n    ', inputSchema={'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'lookup_share_priceArguments', 'type': 'object'}, annotations=None)]

### Vamos testar!

Tomara que o gpt-4o-mini seja esperto o bastante para saber que o simbolo da Apple e AAPL


In [26]:
instructions = "Voce responde perguntas sobre o mercado de acoes."
request = "Qual e o preco da acao da Apple?"
model = "gpt-5-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


O preço atual da ação da Apple (AAPL) é US$41,00.

Observação: esse valor está em dólares americanos e pode estar sujeito a atraso ou variação rápida. Deseja que eu converta para reais, mostre o histórico intradiário/últimos dias, ou confirme um preço em tempo real?

## Polygon.io Parte 2: plano pago - totalmente opcional!

Se tiver interesse, voce pode assinar o plano mensal para obter dados de mercado mais atualizados e chamadas de API ilimitadas.

Se quiser fazer isso, tambem faz sentido usar o servidor MCP completo que a Polygon.io lancou, para aproveitar toda a funcionalidade deles.



In [ ]:

params = {"command": "uvx",
          "args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@v0.1.0", "mcp_polygon"],
          "env": {"POLYGON_API_KEY": polygon_api_key}
          }
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools


### Uau, isso e um monte de ferramentas!

Vamos experimentar - tomara que a quantidade de ferramentas nao deixe o gpt-4o-mini sobrecarregado!

Com o plano mensal de US$29, nao temos acesso a algumas das APIs, entao precisei especificar quais APIs podem ser chamadas.

Se voce investiu em um plano maior, fique a vontade para remover minha restricao extra..


In [ ]:
instructions = "Voce responde perguntas sobre o mercado de acoes."
request = "Qual e o preco da acao da Apple? Use a sua ferramenta get_snapshot_ticker para obter o preco mais recente."
model = "gpt-5-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## Configurando o seu arquivo .env

Se decidir ter um plano pago, adicione isto ao seu arquivo .env para indicar:

`POLYGON_PLAN=paid`

E se decidir ir ate o fim pela API em tempo real, entao faca:

`POLYGON_PLAN=realtime`


In [ ]:
load_dotenv(override=True)

polygon_plan = os.getenv("POLYGON_PLAN")
is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

if is_paid_polygon:
    print("Voce escolheu assinar o plano pago da Polygon, entao o codigo vai olhar precos com atraso de 15 min")
elif is_realtime_polygon:
    print("Uau - voce escolheu assinar o plano em tempo real da Polygon, entao o codigo vai olhar precos em tempo real")
else:
    print("De acordo com o seu arquivo .env, voce escolheu assinar o plano gratuito da Polygon, entao o codigo vai olhar precos de fim de dia")


## E isso e tudo por hoje!

Removi a parte deste laboratorio que usa o servidor mcp "Financial Datasets", porque ele e inferior - mais caro com menos APIs.

E assim conseguimos usar o mesmo provedor para APIs gratuitas e pagas.

Mas se quiser ver o codigo, basta olhar no historico do git por uma versao anterior.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercicios</h2>
            <span style="color:#ff7800;">Explore os marketplaces de servidores MCP e integre o seu proprio, usando todas as 3 abordagens.
            </span>
        </td>
    </tr>
</table>